# Practical 8: Data Visualization and Exploratory Data Analysis (EDA)
**Course:** Python for Data Science (`BE05000231_PDS`)  
**Semester:** 5th Semester B.E. / B.Tech  
**Topic:** Advanced Security Telemetry Visualization with Matplotlib, Seaborn, and Plotly  
**Pipeline Input:** Practical 7 Clean Wrangled Telemetry (`wrangled_filtered_dataset.parquet` — 160,076 records)  

---

## 1. AIM & OBJECTIVES
### Aim
To use visual tools (Matplotlib, Seaborn, Plotly) to understand cybersecurity telemetry by constructing requests per hour time-series curves, top attacking IP rankings, temporal attack category distributions, HTTP status code breakdowns, and IP vs request method crosstab heatmaps.

### Objectives
1. **Pipeline Continuity:** Ingest the clean wrangled dataset from Practical 7 (`wrangled_filtered_dataset.parquet`).
2. **Requests per Hour:** Visualize time-series arrival velocity and malicious attack surges.
3. **Top 10 Attacking IPs:** Plot persistent threat actor rankings using Seaborn horizontal bar charts.
4. **Attack Categories Over Time:** Map the multi-vector temporal progression of attacks using multi-line and stacked area charts.
5. **Status Code Distributions:** Evaluate HTTP server feedback (2xx, 3xx, 4xx, 5xx) using bar charts and donut charts.
6. **Heatmap of IP vs Request Types:** Cross-tabulate top client IPs against HTTP methods (GET, POST, HEAD, PUT).
7. **Interactive Dashboarding:** Export standalone Plotly HTML figures with hover metadata and range sliders.

---
## 2. SETUP & ENVIRONMENT
We import `pandas`, `numpy`, `matplotlib`, `seaborn`, and `plotly`.

In [ ]:
import sys
import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

# Resolve directories
PRACTICAL_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
WORKSPACE_DIR = PRACTICAL_DIR.parent
P7_PROCESSED = WORKSPACE_DIR / 'Practical 7' / 'data' / 'processed'
PLOTS_DIR = PRACTICAL_DIR / 'outputs' / 'plots'
INTERACTIVE_DIR = PRACTICAL_DIR / 'outputs' / 'interactive'

PLOTS_DIR.mkdir(parents=True, exist_ok=True)
INTERACTIVE_DIR.mkdir(parents=True, exist_ok=True)

print("Environment initialized successfully.")

---
## 3. INGESTION & DATA INSPECTION
We load `wrangled_filtered_dataset.parquet` generated in Practical 7.

In [ ]:
pq_path = P7_PROCESSED / 'wrangled_filtered_dataset.parquet'
df = pd.read_parquet(pq_path)
df['datetime_utc'] = pd.to_datetime(df['datetime_utc'], errors='coerce')

print(f"Ingested Telemetry: {len(df):,} records, {len(df.columns)} columns")
print(f"Unique Client IPs: {df['client_ip'].nunique():,}")
print(f"Classes Present: {df['label'].nunique()}")

---
## 4. PLOT 1 — REQUESTS PER HOUR (TIME-SERIES)
Hourly arrival velocity of total requests vs malicious attacks.

In [ ]:
df_ts = df.dropna(subset=['datetime_utc']).sort_values('datetime_utc').set_index('datetime_utc')
hourly = df_ts.resample('1h').agg(
    total_requests=('line_number', 'count'),
    attacks=('is_attack', 'sum')
).reset_index()

plt.figure(figsize=(12, 4.5), dpi=150)
plt.plot(hourly['datetime_utc'], hourly['total_requests'], color='#2563EB', linewidth=2.0, label='Total Requests')
plt.plot(hourly['datetime_utc'], hourly['attacks'], color='#EF4444', linewidth=2.2, label='Malicious Attacks')
plt.fill_between(hourly['datetime_utc'], 0, hourly['attacks'], color='#EF4444', alpha=0.25)
plt.title('Hourly Access Log Traffic Velocity & Attack Bursts', fontsize=12, fontweight='bold')
plt.ylabel('Requests / Hour')
plt.xlabel('Time (UTC)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

---
## 5. PLOT 2 — TOP 10 ATTACKING IPS (FREQUENCY ANALYSIS)
Identifies persistent malicious threat actors.

In [ ]:
ip_stats = df.groupby('client_ip')['is_attack'].sum().reset_index()
top10 = ip_stats.sort_values(by='is_attack', ascending=False).head(10).iloc[::-1]

plt.figure(figsize=(10, 4.5), dpi=150)
sns.barplot(data=top10, y='client_ip', x='is_attack', color='#DC2626', edgecolor='#7F1D1D')
plt.title('Top 10 Persistent Malicious Client IP Addresses', fontsize=12, fontweight='bold')
plt.xlabel('Attack Events')
plt.ylabel('Client IP Address')
plt.grid(True, axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

---
## 6. PLOT 3 — ATTACK CATEGORIES OVER TIME
Temporal progression of specific attack classes.

In [ ]:
attacks_only = df[df['label'] != 'benign'].dropna(subset=['datetime_utc'])
temporal_cats = attacks_only.groupby([pd.Grouper(key='datetime_utc', freq='2h'), 'label'])['line_number'].count().unstack(fill_value=0)

plt.figure(figsize=(12, 5), dpi=150)
for col in temporal_cats.columns:
    plt.plot(temporal_cats.index, temporal_cats[col], label=col, linewidth=1.8)

plt.title('Temporal Dynamics of Web Threat Vectors over Time (2-Hour Bins)', fontsize=12, fontweight='bold')
plt.ylabel('Attacks / Interval')
plt.xlabel('Time (UTC)')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

---
## 7. PLOT 4 — STATUS CODE DISTRIBUTIONS
HTTP status code frequencies and response classes (2xx, 3xx, 4xx, 5xx).

In [ ]:
status_counts = df['status_code'].value_counts().reset_index().head(8)
status_counts.columns = ['status_code', 'count']
status_counts['status_str'] = status_counts['status_code'].astype(str)

plt.figure(figsize=(8, 4), dpi=150)
palette = ['#10B981' if s < 300 else '#F59E0B' if s < 400 else '#EF4444' for s in status_counts['status_code']]
sns.barplot(data=status_counts, x='status_str', y='count', hue='status_str', palette=palette, legend=False, edgecolor='#1E293B')
plt.title('HTTP Response Status Code Frequencies', fontsize=12, fontweight='bold')
plt.xlabel('HTTP Status Code')
plt.ylabel('Count')
plt.grid(True, axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

---
## 8. PLOT 5 — HEATMAP OF IP VS REQUEST TYPES
Cross-tabulating top 20 client IPs against HTTP request methods.

In [ ]:
req_col = 'request_method' if 'request_method' in df.columns else 'method'
top_ips = df['client_ip'].value_counts().head(20).index
crosstab_ip_req = pd.crosstab(df[df['client_ip'].isin(top_ips)]['client_ip'], df[df['client_ip'].isin(top_ips)][req_col])

plt.figure(figsize=(10, 6.5), dpi=150)
sns.heatmap(crosstab_ip_req, annot=True, fmt='d', cmap='YlOrRd', linewidths=0.5, cbar_kws={'label': 'Request Count'})
plt.title('Cross-Tabulation Heatmap: Top 20 Client IPs vs HTTP Request Methods', fontsize=12, fontweight='bold')
plt.xlabel('HTTP Request Method')
plt.ylabel('Client IP Address')
plt.tight_layout()
plt.show()